In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.datasets import make_classification

# -------------------------------
# Create synthetic data
# -------------------------------
X, y = make_classification(
    n_samples=900, n_features=10, n_classes=2, random_state=42
)

# Split data among 3 clients
client_X = np.array_split(X, 3)
client_y = np.array_split(y, 3)

# -------------------------------
# Define model
# -------------------------------
class LogisticModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

# -------------------------------
# Local training on client
# -------------------------------
def train_client(model, X, y, epochs=3):
    criterion = nn.BCELoss()
    optimizer = optim.SGD(model.parameters(), lr=0.01)

    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

    for _ in range(epochs):
        optimizer.zero_grad()
        output = model(X)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()

    return model.state_dict()

# -------------------------------
# Federated Averaging
# -------------------------------
def fed_avg(weights):
    avg_weights = {}
    for key in weights[0].keys():
        avg_weights[key] = sum(w[key] for w in weights) / len(weights)
    return avg_weights

# -------------------------------
# Federated Learning process
# -------------------------------
global_model = LogisticModel()
rounds = 5

for r in range(rounds):
    local_weights = []

    for i in range(3):
        local_model = LogisticModel()
        local_model.load_state_dict(global_model.state_dict())

        w = train_client(local_model, client_X[i], client_y[i])
        local_weights.append(w)

    global_model.load_state_dict(fed_avg(local_weights))
    print(f"Round {r+1} completed")

# -------------------------------
# Evaluation
# -------------------------------
X_test, y_test = make_classification(
    n_samples=300, n_features=10, n_classes=2, random_state=1
)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

with torch.no_grad():
    preds = global_model(X_test)
    accuracy = ((preds > 0.5) == y_test).float().mean()

print("Final Accuracy:", accuracy.item())


Round 1 completed
Round 2 completed
Round 3 completed
Round 4 completed
Round 5 completed
Final Accuracy: 0.38999998569488525
